In [ ]:

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.utils.data import DataLoader, Dataset

from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from tqdm import tqdm

from google.colab import drive

import warnings
import math
import numbers

random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# mount Google Drive
drive.mount('/content/drive')

# paths to PhysioNet 2012 data in Google Drive
google_drive_folder = '/content/drive/MyDrive/physionet2012/'
set_a_directory = f"{google_drive_folder}/set-a"
set_b_directory = f"{google_drive_folder}/set-b"
set_c_directory = f"{google_drive_folder}/set-c"
outcomes_a_file = f"{set_a_directory}/Outcomes-a.txt"
outcomes_b_file = f"{set_b_directory}/Outcomes-b.txt"
outcomes_c_file = f"{set_c_directory}/Outcomes-c.txt"

print("Set A Directory:", set_a_directory)
print("Set B Directory:", set_b_directory)
print("Set C Directory:", set_c_directory)
print("Outcomes A File:", outcomes_a_file)
print("Outcomes B File:", outcomes_b_file)
print("Outcomes C File:", outcomes_c_file)

Mounted at /content/drive
Set A Directory: /content/drive/MyDrive/physionet2012//set-a
Set B Directory: /content/drive/MyDrive/physionet2012//set-b
Set C Directory: /content/drive/MyDrive/physionet2012//set-c
Outcomes A File: /content/drive/MyDrive/physionet2012//set-a/Outcomes-a.txt
Outcomes B File: /content/drive/MyDrive/physionet2012//set-b/Outcomes-b.txt
Outcomes C File: /content/drive/MyDrive/physionet2012//set-c/Outcomes-c.txt


In [ ]:
import os
import random
import numpy as np
import pandas as pd


random.seed(42)
torch.manual_seed(42)
np.random.seed(42)


google_drive_folder = '/content/drive/MyDrive/physionet2012/'

set_a_directory = os.path.join(google_drive_folder, "set-a")


print(f"Using directory: '{set_a_directory}'")
print("Please ensure this path is correct for your environment.")


dynamic_features_list = [
    'Albumin', 'ALP', 'ALT', 'AST', 'Bilirubin', 'BUN', 'Cholesterol',
    'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT',
    'HR', 'K', 'Lactate', 'Mg', 'MAP', 'MechVent', 'Na', 'NIDiasABP',
    'NIMAP', 'NISysABP', 'PaCO2', 'PaO2', 'pH', 'Platelets', 'RespRate',
    'SaO2', 'SysABP', 'Temp', 'TropI', 'TropT', 'Urine', 'WBC', 'Weight'
]
dynamic_features_set = set(dynamic_features_list)

max_common_features = 0
patient_feature_counts = {}

try:
    if not os.path.exists(set_a_directory):
        raise FileNotFoundError(f"Directory not found: {set_a_directory}")

    all_files = os.listdir(set_a_directory)
    patient_files = [f for f in all_files if f.endswith('.txt') and not f.startswith('Outcomes')]
    print(f"Found {len(patient_files)} patient files.")

    if not patient_files:
        print("Error: No patient files found in the directory.")
    else:
        print("Processing patient files...")
        processed_count = 0
        for filename in patient_files:
            file_path = os.path.join(set_a_directory, filename)
            record_id_str = filename.replace('.txt', '')

            try:
                df = pd.read_csv(file_path)

                if 'Parameter' in df.columns:
                    features_in_file = set(df['Parameter'].unique())

                    common_features = dynamic_features_set.intersection(features_in_file)

                    current_common_features = len(common_features)
                    patient_feature_counts[record_id_str] = current_common_features

                    if current_common_features > max_common_features:
                        max_common_features = current_common_features
                    processed_count += 1
                else:
                    print(f"Warning: 'Parameter' column not found in {filename}. Skipping.")

            except pd.errors.EmptyDataError:
                print(f"Warning: File {filename} is empty. Skipping.")
            except Exception as e:
                print(f"Error processing file {filename}: {e}")

        print(f"\nProcessed {processed_count} files.")
        print(f"Maximum number of common dynamic features found across all patients in set-a: {max_common_features}")

        if max_common_features == 34:
            print("\nThis result (34) matches the feature dimension used in the GRU_D_physionet.ipynb notebook.")
            print("This confirms the theory that the input dimension was derived dynamically based on the maximum features found in any single patient file in set-a.")
        else:
             print(f"\nThis result ({max_common_features}) does NOT match the expected feature dimension (34).")
             print("There might be an issue with the path, the dynamic_features list definition, or the notebook's original calculation.")

except FileNotFoundError as fnf_error:
    print(f"\n{fnf_error}")
    print("Please verify the path to your 'set-a' directory and try again.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Using directory: '/content/drive/MyDrive/physionet2012/set-a'
Please ensure this path is correct for your environment.
Found 4000 patient files.
Processing patient files...

Processed 4000 files.
Maximum number of common dynamic features found across all patients in set-a: 34

This result (34) matches the feature dimension used in the GRU_D_physionet.ipynb notebook.
This confirms the theory that the input dimension was derived dynamically based on the maximum features found in any single patient file in set-a.


In [ ]:
all_dynamic_features_list = [
    'Albumin', 'ALP', 'ALT', 'AST', 'Bilirubin', 'BUN', 'Cholesterol',
    'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT',
    'HR', 'K', 'Lactate', 'Mg', 'MAP', 'MechVent', 'Na', 'NIDiasABP',
    'NIMAP', 'NISysABP', 'PaCO2', 'PaO2', 'pH', 'Platelets', 'RespRate',
    'SaO2', 'SysABP', 'Temp', 'TropI', 'TropT', 'Urine', 'WBC', 'Weight'
]
all_dynamic_features_set = set(all_dynamic_features_list)
print(f"Total potential dynamic features defined: {len(all_dynamic_features_set)}")

# --- Step 1: Find the set of features ACTUALLY present in set-a files ---

features_found_in_set_a = set()

try:
    if not os.path.exists(set_a_directory):
        raise FileNotFoundError(f"Directory not found: {set_a_directory}")

    all_files = os.listdir(set_a_directory)
    patient_files = [f for f in all_files if f.endswith('.txt') and not f.startswith('Outcomes')]
    print(f"Found {len(patient_files)} patient files. Analyzing features present...")

    if not patient_files:
        print("Error: No patient files found.")
    else:
        for filename in patient_files:
            file_path = os.path.join(set_a_directory, filename)
            try:
                df = pd.read_csv(file_path)
                if 'Parameter' in df.columns:
                    features_found_in_set_a.update(df['Parameter'].unique())
                else:
                     print(f"Warning: 'Parameter' column missing in {filename}")

            except pd.errors.EmptyDataError:
                print(f"Warning: File {filename} is empty. Skipping.")
            except Exception as e:
                print(f"Error processing file {filename} for feature discovery: {e}")

        # --- Step 2: Identify the features used by the model (intersection) ---
        common_dynamic_features_set = all_dynamic_features_set.intersection(features_found_in_set_a)
        print(f"\nTotal unique dynamic features found across set-a AND in the defined list: {len(common_dynamic_features_set)}")

        # --- Step 3: Identify the dropped features ---
        dropped_features = all_dynamic_features_set - common_dynamic_features_set
        print(f"Features defined but NOT found commonly across set-a (dropped): {len(dropped_features)}")
        if dropped_features:
            print(f"  -> {sorted(list(dropped_features))}")
        else:
            print("  -> None (All defined dynamic features were found in the common set)")

        # --- Step 4: Check if any dropped features exist in ANY file ---
        found_dropped_features = set()
        if dropped_features:
            print("\nChecking if any of the dropped features appear in individual files...")
            for filename in patient_files:
                file_path = os.path.join(set_a_directory, filename)
                try:
                    df = pd.read_csv(file_path)
                    if 'Parameter' in df.columns:
                        features_in_this_file = set(df['Parameter'].unique())
                        found_in_this_file = dropped_features.intersection(features_in_this_file)
                        if found_in_this_file:
                            found_dropped_features.update(found_in_this_file)

                except pd.errors.EmptyDataError:
                    pass
                except Exception as e:
                    print(f"Error processing file {filename} during dropped feature check: {e}")

            # --- Step 5: Report Findings ---
            print("\n--- Final Report ---")
            if not dropped_features:
                 print("No features were identified as dropped during preprocessing.")
            elif not found_dropped_features:
                print("None of the dropped features were found in any individual patient record in set-a.")
                print(f"Dropped features list: {sorted(list(dropped_features))}")
            else:
                print("The following dropped features WERE FOUND in at least one patient record in set-a:")
                for feature in sorted(list(found_dropped_features)):
                    print(f"  - {feature}")

                features_dropped_but_never_seen = dropped_features - found_dropped_features
                if features_dropped_but_never_seen:
                    print("\nThe following dropped features were NOT FOUND in any individual record in set-a:")
                    for feature in sorted(list(features_dropped_but_never_seen)):
                         print(f"  - {feature}")
        else:
             print("\nNo features were dropped, so no check needed for their existence.")


except FileNotFoundError as fnf_error:
    print(f"\n{fnf_error}")
    print("Please verify the path to your 'set-a' directory and try again.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Total potential dynamic features defined: 37
Found 4000 patient files. Analyzing features present...

Total unique dynamic features found across set-a AND in the defined list: 35
Features defined but NOT found commonly across set-a (dropped): 2
  -> ['TropI', 'TropT']

Checking if any of the dropped features appear in individual files...

--- Final Report ---
None of the dropped features were found in any individual patient record in set-a.
Dropped features list: ['TropI', 'TropT']


In [ ]:
import os
import pandas as pd
import numpy as np

# --- Configuration ---

google_drive_folder = '/content/drive/MyDrive/physionet2012/'
set_a_directory = os.path.join(google_drive_folder, "set-a")

print(f"Using directory: '{set_a_directory}'")

all_dynamic_features_list = [
    'Albumin', 'ALP', 'ALT', 'AST', 'Bilirubin', 'BUN', 'Cholesterol',
    'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT',
    'HR', 'K', 'Lactate', 'Mg', 'MAP', 'MechVent', 'Na', 'NIDiasABP',
    'NIMAP', 'NISysABP', 'PaCO2', 'PaO2', 'pH', 'Platelets', 'RespRate',
    'SaO2', 'SysABP', 'Temp', 'TropI', 'TropT', 'Urine', 'WBC', 'Weight'
]
all_dynamic_features_set = set(all_dynamic_features_list)
print(f"Total potential dynamic features defined: {len(all_dynamic_features_set)}")

# --- Verification Logic ---

features_found_union = set()
max_features_in_single_file = 0
patient_max_features_id = None
features_in_max_patient_file = set()

try:
    if not os.path.exists(set_a_directory):
        raise FileNotFoundError(f"Directory not found: {set_a_directory}")

    all_files = os.listdir(set_a_directory)
    patient_files = [f for f in all_files if f.endswith('.txt') and not f.startswith('Outcomes')]
    print(f"Found {len(patient_files)} patient files. Analyzing features...")

    if not patient_files:
        print("Error: No patient files found.")
    else:
        processed_count = 0
        for filename in patient_files:
            file_path = os.path.join(set_a_directory, filename)
            record_id_str = filename.replace('.txt', '')

            try:
                df = pd.read_csv(file_path)
                if 'Parameter' in df.columns:
                    features_in_this_file = set(df['Parameter'].unique())

                    features_found_union.update(features_in_this_file)

                    common_features_this_file = all_dynamic_features_set.intersection(features_in_this_file)

                    current_common_features = len(common_features_this_file)

                    if current_common_features > max_features_in_single_file:
                        max_features_in_single_file = current_common_features
                        patient_max_features_id = record_id_str
                        features_in_max_patient_file = common_features_this_file

                    processed_count += 1
                else:
                     print(f"Warning: 'Parameter' column missing in {filename}")

            except pd.errors.EmptyDataError:
                print(f"Warning: File {filename} is empty. Skipping.")
            except Exception as e:
                print(f"Error processing file {filename}: {e}")

        print(f"\nProcessed {processed_count} files.")

        # --- Identify the features used by the model (intersection with UNION) ---
        common_dynamic_features_union = all_dynamic_features_set.intersection(features_found_union)
        print(f"\nTotal unique dynamic features found across ALL set-a files AND in the defined list: {len(common_dynamic_features_union)}")
        print("Common features (Union):")
        print(f"  -> {sorted(list(common_dynamic_features_union))}")

        # --- Identify the dropped features (based on UNION) ---
        dropped_features = all_dynamic_features_set - common_dynamic_features_union
        print(f"\nFeatures defined but NOT found in ANY set-a file (dropped): {len(dropped_features)}")
        if dropped_features:
            print(f"  -> {sorted(list(dropped_features))}")
        else:
            print("  -> None")

        # --- Report the MAX features found in a SINGLE file ---
        print(f"\nMaximum number of common dynamic features found in ANY SINGLE patient file in set-a: {max_features_in_single_file}")
        if patient_max_features_id:
            print(f"(Found in patient record: {patient_max_features_id})")
        else:
            print("(Could not determine patient with max features - possibly no files processed)")


        # --- Final Conclusion and Identification of the 'Odd One Out' ---
        print("\n--- Final Conclusion ---")
        if max_features_in_single_file == 34 and len(common_dynamic_features_union) == 35:
             print("The notebook correctly used an input dimension of 34, based on the maximum features found co-occurring within a single patient.")
             print("One feature exists in the dataset overall but never alongside all others in the file with the maximum count.")

             # Explicitly find and print the difference
             odd_one_out_feature_set = common_dynamic_features_union - features_in_max_patient_file
             if len(odd_one_out_feature_set) == 1:
                 odd_one_out_feature = list(odd_one_out_feature_set)[0]
                 print(f"\nThe feature present in the common set (35) but MISSING from the patient file with the max count ({patient_max_features_id}, count: 34) is: '{odd_one_out_feature}'")
             elif len(odd_one_out_feature_set) > 1:
                 print(f"\nError: Found more than one feature difference between the union set and the max patient set: {sorted(list(odd_one_out_feature_set))}")
             else:
                  print("\nError: Could not find the feature difference. The sets might be identical unexpectedly.")


        elif len(common_dynamic_features_union) == 34 and max_features_in_single_file == 34:
             print("Both methods agree: 34 common features were found, and the maximum in any file was 34.")
             print(f"Features dropped entirely (not found in any file): {sorted(list(dropped_features))}")
        else:
             print(f"Discrepancy found: Total common features = {len(common_dynamic_features_union)}, Max per file = {max_features_in_single_file}. Expected 35 and 34 respectively.")
             print("Please re-verify the feature lists and file processing.")


except FileNotFoundError as fnf_error:
    print(f"\n{fnf_error}")
    print("Please verify the path to your 'set-a' directory and try again.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Using directory: '/content/drive/MyDrive/physionet2012/set-a'
Total potential dynamic features defined: 37
Found 4000 patient files. Analyzing features...

Processed 4000 files.

Total unique dynamic features found across ALL set-a files AND in the defined list: 35
Common features (Union):
  -> ['ALP', 'ALT', 'AST', 'Albumin', 'BUN', 'Bilirubin', 'Cholesterol', 'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT', 'HR', 'K', 'Lactate', 'MAP', 'MechVent', 'Mg', 'NIDiasABP', 'NIMAP', 'NISysABP', 'Na', 'PaCO2', 'PaO2', 'Platelets', 'RespRate', 'SaO2', 'SysABP', 'Temp', 'Urine', 'WBC', 'Weight', 'pH']

Features defined but NOT found in ANY set-a file (dropped): 2
  -> ['TropI', 'TropT']

Maximum number of common dynamic features found in ANY SINGLE patient file in set-a: 34
(Found in patient record: 141577)

--- Final Conclusion ---
The notebook correctly used an input dimension of 34, based on the maximum features found co-occurring within a single patient.
One feature exists 

In [ ]:
import os
import pandas as pd
import numpy as np

# --- Configuration ---

google_drive_folder = '/content/drive/MyDrive/physionet2012/'
set_a_directory = os.path.join(google_drive_folder, "set-a")

print(f"Using directory: '{set_a_directory}'")

all_dynamic_features_list = [
    'Albumin', 'ALP', 'ALT', 'AST', 'Bilirubin', 'BUN', 'Cholesterol',
    'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT',
    'HR', 'K', 'Lactate', 'Mg', 'MAP', 'MechVent', 'Na', 'NIDiasABP',
    'NIMAP', 'NISysABP', 'PaCO2', 'PaO2', 'pH', 'Platelets', 'RespRate',
    'SaO2', 'SysABP', 'Temp', 'TropI', 'TropT', 'Urine', 'WBC', 'Weight'
]
all_dynamic_features_set = set(all_dynamic_features_list)
print(f"Total potential dynamic features defined: {len(all_dynamic_features_set)}")

# --- Verification Logic ---

features_found_union = set()
max_features_in_single_file = 0
patient_max_features_id = None
features_in_max_patient_file = set()

try:
    if not os.path.exists(set_a_directory):
        raise FileNotFoundError(f"Directory not found: {set_a_directory}")

    all_files = os.listdir(set_a_directory)
    patient_files = [f for f in all_files if f.endswith('.txt') and not f.startswith('Outcomes')]
    print(f"Found {len(patient_files)} patient files. Analyzing features...")

    if not patient_files:
        print("Error: No patient files found.")
    else:
        processed_count = 0
        for filename in patient_files:
            file_path = os.path.join(set_a_directory, filename)
            record_id_str = filename.replace('.txt', '')

            try:
                df = pd.read_csv(file_path)
                if 'Parameter' in df.columns:
                    features_in_this_file = set(df['Parameter'].unique())

                    features_found_union.update(features_in_this_file)

                    common_features_this_file = all_dynamic_features_set.intersection(features_in_this_file)

                    current_common_features = len(common_features_this_file)

                    if current_common_features > max_features_in_single_file:
                        max_features_in_single_file = current_common_features
                        patient_max_features_id = record_id_str
                        features_in_max_patient_file = common_features_this_file

                    processed_count += 1
                else:
                     print(f"Warning: 'Parameter' column missing in {filename}")

            except pd.errors.EmptyDataError:
                print(f"Warning: File {filename} is empty. Skipping.")
            except Exception as e:
                print(f"Error processing file {filename}: {e}")

        print(f"\nProcessed {processed_count} files.")

        # --- Identify the features used by the model (intersection with UNION) ---
        common_dynamic_features_union = all_dynamic_features_set.intersection(features_found_union)
        print(f"\nTotal unique dynamic features found across ALL set-a files AND in the defined list: {len(common_dynamic_features_union)}")
        print("Common features (Union):")
        print(f"  -> {sorted(list(common_dynamic_features_union))}")

        # --- Identify the dropped features (based on UNION) ---
        dropped_features = all_dynamic_features_set - common_dynamic_features_union
        print(f"\nFeatures defined but NOT found in ANY set-a file (dropped): {len(dropped_features)}")
        if dropped_features:
            print(f"  -> {sorted(list(dropped_features))}")
        else:
            print("  -> None")

        # --- Report the MAX features found in a SINGLE file ---
        print(f"\nMaximum number of common dynamic features found in ANY SINGLE patient file in set-a: {max_features_in_single_file}")
        if patient_max_features_id:
            print(f"(Found in patient record: {patient_max_features_id})")
        else:
            print("(Could not determine patient with max features - possibly no files processed)")


        # --- Final Conclusion and Identification of the 'Odd One Out' ---
        print("\n--- Final Conclusion ---")
        if max_features_in_single_file == 34 and len(common_dynamic_features_union) == 35:
             print("The notebook correctly used an input dimension of 34, based on the maximum features found co-occurring within a single patient.")
             print("One feature exists in the dataset overall but never alongside all others in the file with the maximum count.")

             # Explicitly find and print the difference
             odd_one_out_feature_set = common_dynamic_features_union - features_in_max_patient_file
             if len(odd_one_out_feature_set) == 1:
                 odd_one_out_feature = list(odd_one_out_feature_set)[0]
                 print(f"\nThe feature present in the common set (35) but MISSING from the patient file with the max count ({patient_max_features_id}, count: 34) is: '{odd_one_out_feature}'")
             elif len(odd_one_out_feature_set) > 1:
                 print(f"\nError: Found more than one feature difference between the union set and the max patient set: {sorted(list(odd_one_out_feature_set))}")
             else:
                  print("\nError: Could not find the feature difference. The sets might be identical unexpectedly.")


        elif len(common_dynamic_features_union) == 34 and max_features_in_single_file == 34:
             print("Both methods agree: 34 common features were found, and the maximum in any file was 34.")
             print(f"Features dropped entirely (not found in any file): {sorted(list(dropped_features))}")
        else:
             print(f"Discrepancy found: Total common features = {len(common_dynamic_features_union)}, Max per file = {max_features_in_single_file}. Expected 35 and 34 respectively.")
             print("Please re-verify the feature lists and file processing.")


except FileNotFoundError as fnf_error:
    print(f"\n{fnf_error}")
    print("Please verify the path to your 'set-a' directory and try again.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Using directory: '/content/drive/MyDrive/physionet2012/set-a'
Total potential dynamic features defined: 37
Found 4000 patient files. Analyzing features...

Processed 4000 files.

Total unique dynamic features found across ALL set-a files AND in the defined list: 35
Common features (Union):
  -> ['ALP', 'ALT', 'AST', 'Albumin', 'BUN', 'Bilirubin', 'Cholesterol', 'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT', 'HR', 'K', 'Lactate', 'MAP', 'MechVent', 'Mg', 'NIDiasABP', 'NIMAP', 'NISysABP', 'Na', 'PaCO2', 'PaO2', 'Platelets', 'RespRate', 'SaO2', 'SysABP', 'Temp', 'Urine', 'WBC', 'Weight', 'pH']

Features defined but NOT found in ANY set-a file (dropped): 2
  -> ['TropI', 'TropT']

Maximum number of common dynamic features found in ANY SINGLE patient file in set-a: 34
(Found in patient record: 141577)

--- Final Conclusion ---
The notebook correctly used an input dimension of 34, based on the maximum features found co-occurring within a single patient.
One feature exists 